In [3]:
import pandas as pd
import re

In [4]:
INPUT_FILE = "raw_data/job_postings.csv"
OUTPUT_FILE = "output/clean_jobs.csv"

In [5]:
df = pd.read_csv(INPUT_FILE)
print("Total raws:",len(df))

Total raws: 15886


In [6]:
TITLE_KEYWORDS = ["data analyst", "business analyst", "data scientist", "bi developer", "data engineer", "analytics"]
pattern = "|".join(TITLE_KEYWORDS)
mask = df["title"].str.lower().str.contains(pattern, na=False, regex=True)
df = df[mask]
print("Rows after filtering:", len(df))

Rows after filtering: 337


In [13]:
def clean_text(x):
    if pd.isna(x):
        return x
    x = str(x)
    x = re.sub(r"\s+", " ", x)
    return x.strip()

df["title"] = df["title"].apply(clean_text)
df["location"] = df["location"].apply(clean_text)
df["description"] = df["description"].apply(clean_text)

print("Text cleaned")

Text cleaned


In [14]:
df["salary_numeric"] = df[["min_salary", "max_salary"]].mean(axis=1)

def annualize(row):
    if pd.isna(row["salary_numeric"]):
        return None
    period = str(row["pay_period"]).upper()
    if period == "HOURLY":
        return row["salary_numeric"] * 40 * 52
    if period == "MONTHLY":
        return row["salary_numeric"] * 12
    if period == "WEEKLY":
        return row["salary_numeric"] * 52
    return row["salary_numeric"]

df["salary_annual"] = df.apply(annualize, axis=1)
df["posted_date"] = pd.to_datetime(df["listed_time"], unit="ms", errors="coerce")

print("Salary and date fixed")

Salary and date fixed


In [16]:
before = len(df)
df = df.drop_duplicates(subset=["title", "location", "description"])
print("Removed duplicates:", before - len(df))

df = df[df["description"].notna() & (df["description"].str.len() > 20)]

df.to_csv(OUTPUT_FILE, index=False)
print("Final cleaned rows:", len(df))
print("Saved to:", OUTPUT_FILE)

Removed duplicates: 0
Final cleaned rows: 336
Saved to: output/clean_jobs.csv


In [17]:
SKILL_LIST = ["Python", "SQL", "Power BI", "Tableau", "Excel", "Machine Learning",
              "Statistics", "ETL", "Communication", "R", "AWS", "Azure"]

def extract_skills(text):
    if pd.isna(text):
        return []
    text_lower = text.lower()
    found = []
    for skill in SKILL_LIST:
        pattern = r"\b" + re.escape(skill.lower()) + r"\b"
        if re.search(pattern, text_lower):
            found.append(skill)
    return found

df["skills_found"] = df["description"].apply(extract_skills)
df["skill_count"] = df["skills_found"].apply(len)

print(df[["title", "skills_found"]].head(10))

                                                 title  \
203                        Enterprise Business Analyst   
266                                      Data Engineer   
292                    Data Scientist/ Product Analyst   
313                          Data Analytics Consultant   
354  NEW & REMOTE CONTRACT - PRINCIPAL CONSULTANT /...   
355           IBM TM1 / Planning Analytics Consultants   
396                                Azure Data Engineer   
440                               Senior Data Engineer   
523                               Senior Data Engineer   
536                                   Business Analyst   

                                 skills_found  
203                           [Communication]  
266                             [Python, SQL]  
292  [Python, SQL, Statistics, Communication]  
313           [Power BI, Tableau, ETL, Azure]  
354                                        []  
355                           [Communication]  
396                      

In [18]:
all_skills = []
for skills_list in df["skills_found"]:
    for skill in skills_list:
        all_skills.append(skill)

skill_counts = pd.Series(all_skills).value_counts()
print(skill_counts)

SQL                 157
Communication       144
Python              108
Excel                77
Tableau              67
ETL                  62
Azure                50
Statistics           48
R                    48
AWS                  46
Power BI             41
Machine Learning     37
Name: count, dtype: int64


In [19]:
long_rows = []
for index, row in df.iterrows():
    for skill in row["skills_found"]:
        long_rows.append({
            "title": row["title"],
            "location": row["location"],
            "posted_date": row["posted_date"],
            "salary_annual": row["salary_annual"],
            "skill": skill
        })

skills_long = pd.DataFrame(long_rows)
skills_long.to_csv("output/skills_long.csv", index=False)
print("Saved", len(skills_long), "rows to output/skills_long.csv")
print(skills_long.head())

Saved 885 rows to output/skills_long.csv
                             title                   location  \
0      Enterprise Business Analyst  Greater Pittsburgh Region   
1                    Data Engineer              United States   
2                    Data Engineer              United States   
3  Data Scientist/ Product Analyst          San Francisco, CA   
4  Data Scientist/ Product Analyst          San Francisco, CA   

          posted_date  salary_annual          skill  
0 2023-07-22 04:26:40            NaN  Communication  
1 2023-07-22 04:26:40            NaN         Python  
2 2023-07-22 04:26:40            NaN            SQL  
3 2023-07-22 04:26:40       156000.0         Python  
4 2023-07-22 04:26:40       156000.0            SQL  


In [20]:
TAUGHT_SKILLS = ["Excel", "SQL", "Python", "Power BI", "Statistics", "Communication"]

demand = skills_long["skill"].value_counts().reset_index()
demand.columns = ["skill", "demand_count"]

total_jobs = df.shape[0]
demand["demand_pct"] = (demand["demand_count"] / total_jobs * 100).round(1)

demand["taught_in_course"] = demand["skill"].isin(TAUGHT_SKILLS)
demand["gap_flag"] = demand["taught_in_course"].apply(lambda x: "Covered" if x else "Gap - not taught")

demand = demand.sort_values("demand_count", ascending=False)
demand.to_csv("output/skill_gap.csv", index=False)

print(demand)

               skill  demand_count  demand_pct  taught_in_course  \
0                SQL           157        46.7              True   
1      Communication           144        42.9              True   
2             Python           108        32.1              True   
3              Excel            77        22.9              True   
4            Tableau            67        19.9             False   
5                ETL            62        18.5             False   
6              Azure            50        14.9             False   
7         Statistics            48        14.3              True   
8                  R            48        14.3             False   
9                AWS            46        13.7             False   
10          Power BI            41        12.2              True   
11  Machine Learning            37        11.0             False   

            gap_flag  
0            Covered  
1            Covered  
2            Covered  
3            Covered  


In [21]:
def categorize_title(title):
    title_lower = title.lower()
    if "data scientist" in title_lower:
        return "Data Scientist"
    elif "data engineer" in title_lower:
        return "Data Engineer"
    elif "business analyst" in title_lower:
        return "Business Analyst"
    elif "data analyst" in title_lower:
        return "Data Analyst"
    elif "bi developer" in title_lower or "business intelligence" in title_lower:
        return "BI Developer"
    else:
        return "Other Analytics Role"

df["role_category"] = df["title"].apply(categorize_title)
print(df["role_category"].value_counts())

role_category
Business Analyst        105
Other Analytics Role     77
Data Engineer            63
Data Analyst             42
Data Scientist           34
BI Developer             15
Name: count, dtype: int64
